In [26]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

gpt_api = os.getenv("GPT_API_KEY") 

In [27]:
prompt = """
너는 기업 사업보고서의 "주요 제품", "주요 제품 및 서비스", "주요제품등의 현황" 표에서
매출 구성(금액/비중)을 정확히 추출한다.

절대 원칙
- 표에 없는 숫자 생성 금지
- 다른 행 숫자 복사 금지
- 숫자는 해당 행 셀에서만 읽기

────────────────
1. 컬럼 식별
────────────────
표에서 다음 역할을 찾는다 (순서 무관)

사업부문 (있으면 사용)
매출유형 (제품 / 상품 / 기타 등)
품목 (제품명 / 서비스명)
구체적용도 (품목 보조)
주요상표 (품목 보조)
매출액
비율(%)

────────────────
2. 품목 결정
────────────────
항목 키는 반드시 품목.

품목이 비어있으면 다음 순서로 복원:
1 구체적용도
2 주요상표
3 병합된 직전 품목
직전 품목 계승은
셀 병합이 명확히 확인될 때만 허용한다.

행이 바뀌었거나
매출유형이 바뀌었으면
품목 계승 금지.

그래도 없으면 해당 행 제외.

"기타"는 매출유형이 아니라 품목일 수 있음 → 품목으로 유지.

────────────────
3. 병합셀 처리
────────────────
텍스트만 계승 가능.

사업부문 비어있으면 직전 사업부문 계승
매출유형 비어있으면 직전 매출유형 계승

단, 사업부문 변경 시 초기화.

숫자(금액/비율)는 절대 계승 금지.

────────────────
4. 값 추출
────────────────
금액 공란 → 0
비율 공란 → "0.00%"

금액은 정수
비율은 소수 둘째자리

────────────────
5. 합계
────────────────
합계 행은 별도 저장.
여러 소계 존재 시 최종 합계만.

합계 없으면 금액 합으로 계산.

────────────────
6. 기간 식별
────────────────

표에서 기간 정보를 우선적으로 식별한다.

기간 식별 우선순위:

1순위: 기수 정보 (예: 제23기, 제25기 등)
→ 여러 기수 존재하면 가장 큰 기수 사용

2순위: 연도 정보 (예: 2024년도, 2023년 등)
→ 여러 연도 존재하면 가장 최신 연도 사용

3순위: 기간 정보 없음
→ "기간미상" 사용

주의:
"연결재무제표 기준", "별도기준" 등은
회계 기준일 뿐 기간이 아니다.
기간으로 사용하지 않는다.

연도 범위(YYYY.MM.DD~YYYY.MM.DD)는 종료연도 사용

────────────────
7. 중요 구조 규칙
────────────────
매출유형은 합계 계산 단위가 아니다.
모든 품목은 하나의 전체 매출 집합에 속한다.

즉:
매출유형별 부분 합계는 계산하지 않는다.
전체 품목을 합친 하나의 전체 합계만 존재한다.

매출유형은 단순 분류 계층이다.
────────────────
8. 출력 (JSON만)
────────────────
{
 "<기간>": {
   "<매출유형>": {
     "<품목>": {"금액": 정수, "비중": "0.00%"}
   },
   "합계": {"금액": 정수, "비중": "100.00%"}
 }
}

매출유형은 하위 분류이며
모든 매출유형은 동일한 전체 합계를 공유한다.
매출유형별 개별 합계를 만들지 않는다.

"""

In [28]:
content ="""

2. 주요 제품 및 서비스
(단위: 백만원)


품목
생산(판매)개시일
제22기
제21기
제20기
매출액
비율
매출액
비율
매출액
비율
양배추
2000-08-01
4,801
18.53%
3,834
16.63%
4,847
19.08%
양파
2007-03-26
2,247
8.67%
1,921
8.33%
2,171
8.55%
고추
2006-02-13
1,320
5.09%
1,410
6.12%
1,250
4.92%
무
2005-07-18
2,162
8.34%
1,490
6.46%
1,941
7.64%
옥수수
2000-08-01
1,115
4.30%
960
4.16%
1,011
3.98%
청경채
2015-02-10
494
1.91%
763
3.31%
1,103
4.34%
파
2004-07-21
861
3.32%
398
1.73%
469
1.85%
기타작물
2006-03-02
10,755
41.50%
10,089
43.75%
11,122
43.79%
기타상품
새싹재배기 외
2,159
8.33%
2,194
9.51%
1,486
5.85%
합 계
계
25,914
100.00%
23,059
100.00%
25,400
100.00%
반품추정부채 및 영업외수익 계정대체
-147
 - 
83
 - 
-11
 - 
매출액 계
25,767
23,142
25,389

* 아시아종묘 별도 기준



2-2. 주요제품 서비스 및 판매 전략

당사의 국내영업은 8개 지점으로 운영되고 있으며, 별도로 도시농업본부를 운영하고 있습니다. 이러한 영업 조직을 통해 국내 영업 판매 전략은 크게 2가지로 진행됩니다.

2-2-1. 국내 영업
첫째, 전국 지점에는 전담직원이 상주하여 신규 거래처를 적극적으로 발굴·확보하고 있으며, 기존 거래처를 정기적으로 방문하여 당사 개발 신품종을 홍보하고 주문을 유도함으로써 매출 확대에 기여하고 있습니다. 아울러 체계적인 거래처 관리와 맞춤형 지원 프로그램을 운영함으로써, 전국적으로 시장 점유율을 공격적으로 확대해 나가고 있습니다.

둘째, 도시농업본부는 내방 고객 및 전화 상담을 통해 종자의 특성과 재배 방법을 상세히 안내하고 있으며, 이로 인한 고객 만족도 향상과 매출 증가 효과를 거두고 있습니다. 동시에 도시농업과 가정원예 시장의 성장에 대응하여 온라인마케팅 시스템을 구축, 새로운 고객층을 지속적으로 발굴하고 있습니다. 오프라인 매장인 도시농업백화점 ‘채가원’ 을 통해 종자부터 비료, 화분, 원예자재, 소도구에 이르기까지 도시농업에 필요한 전 품목을 한곳에서 구매할 수 있는 통합 서비스를 제공하고 있습니다.

(a) 국내영업 매출증대 현황
지난 15년간 당사는 고추, 토마토, 무, 수박, 양파 등 부가가치가 높은 과채류 중심의 품종 개량 및 신품종 개발에 집중해 왔습니다. 이와 같은 지속적인 육종 연구의 결과, 각 작물의 시장 상황에 최적화된 품종이 출시되면서 국내영업부의 매출이 향후 더욱 가파르게 성장할 것으로 예상됩니다.

현재 고추, 단호박, 무, 양파, 배추, 청경채, 수박 등 주요 작물과 식량작물인 옥수수는 지역별 시장에서 좋은 평가를 받고 있으며, 직접적인 매출 향상으로 이어지고 있습니다. 최근에는 고추, 배추, 멜론 등의 신품종이 상인들과 농가의 높은 관심 속에 빠르게 시장에 안착하고 있어 매출 증대에 긍정적인 영향을 주고 있습니다.

더불어, 당사 생명공학육종연구소는 첨단 육종 설비 확충과 작물별 전문 육종 인재 영입을 통해 경쟁력 있는 신품종 개발에 박차를 가하고 있으며, 이를 기반으로 국내 시장 점유율을 지속적으로 확대해 나가고 있습니다. 특히 재배 농가의 수익 증대에 실질적인 도움이 되는 품종 개발을 중점적으로 추진 중입니다.


(b) 국내영업 판매 서비스 전략
당사는 시범 재배(시교)를 통해 지역 특색에 맞는 품종을 개발하고, 이를 중심으로 품종별 단지권을 형성하여 지역 기반의 판매시장을 전략적으로 개척하고 있습니다. 실제로 각 지역에 적합한 단지권이 기존에 형성되었으며, 연작 피해로 인해 대체 작물에 대한 수요가 높아지면서 새로운 단지권이 지속적으로 조성되고 있습니다. 이와 함께 시설재배에 대한 투자와 정부·지자체의 지원도 확산되고 있어, 당사는 이러한 시장 흐름에 발맞춰 전략적인 품종 공급과 단지권 중심의 시장 집중 공략을 강화하고 있습니다.

특히 시교 재배로 생산된 작물은 가락시장 마케터를 통해 유통 단계에서 적절한 가격으로 조율되며, 이 과정에서 품종에 대한 인지도가 높아지고 농가의 품종 전환 및 재배 면적 확대에도 긍정적인 영향을 미치고 있습니다. 이러한 일련의 활동은 판매 실적 향상으로 직결되고 있습니다.

기술센터, 농협, 육종가와의 협력도 당사의 주요 전략 중 하나입니다. 전국 도·시·군 단위의 농업기술원 및 기술센터와 협업하여 지역 특화 품종 개발 및 개량 사업을 추진하고 있으며, 실증시험포장 운영을 통해 품종의 재배 특성과 관리 방법을 지역 농가들과 공유하고 있습니다. 이를 통해 품종의 안정성과 생산성 검증이 가능해지고, 농가의 수익 향샹에도 실질적인 기여를 하고 있습니다.

또한, 작물별 수확 시기에 맞춰 영업지원팀이 주관하는 '품평회' 를 전국 각 지역에서 개최하고 있습니다. 이 행사에서는 농가들이 당사 품종을 직접 눈으로 확인하고, 만져보고, 시식함으로써 품질에 대한 신뢰를 높일 수 있으며, 이는 실질적인 판매 확대 효과로 이어지고 있습니다.

판매 전후 관리(Before Service / After Service) 체계도 철저히 운영되고 있습니다. 작물 파종 전에는 지역농가 및 단지권을 방문하여 품목의 특성과 장점을 설명하고, 파종 후에는 전문 영농기술지도를 통해 농가의 생산성과 품질 향상을 적극적으로 지원하고 있습니다.

향후에는 스마트팜 환경에 적합한 품종 개발에도 역량을 집중할 예정입니다. 스마트팜에 최적화된 고품질 품종을 개발하기 위해 시험 재배와 기술 투자를 지속적으로 확대하고 있으며, 토양중심의 전통농업에서 벗어나 고도화된 시설기반 농업 환경에 적합한 품종을 선제적으로 공급할 계획입니다.


2-2-2. 해외 영업 판매 서비스 전략
당사는 해외에서 양채류 십자화과 양배추와 브로콜리, 케일, 청경채 및 가지과 고추, 특수 박과 단호박, 수박, 멜론 등의 종자를 수출 및 특수 시장개발 선도기업으로 업체로 알려져 있습니다.
또한 세계 채소종자 시장이 기능성과 안전성을 중시하는 추세로 바뀌고 있으며, 친환경 재배가 가능한 다양하며 바이러스 내병성 품종들과 조기 수확이 가능한 어린잎 채소들이 주목받고 있는 점을 중시하고 있습니다. 이에 붉은 베이비리프 채소를 다양하게 육성해 붉은색의 다채, 설채, 청경채, 배추 등의 종자를 영국, 미국, 일본, 호주, 뉴질랜드 등에 수출하였으며, 그 이후 붉은색 계열의 청경채, 소송채, 곱슬케일, 소송채, 경수채 등의 다양한 타입과 신품종을 전 세계 국가에 수출하고 있습니다.
당사 전체 수출액의 약 45% 이상을 점유하고 있는 국가는 중국 및 인도입니다.
이러한 두 나라는 거대한 인구와 종자시장 규모를 가지고 있습니다. 이에 당사는 중국 및 인도의 지역 곳곳을 세부적으로 접근하는 영업 정책을 펴 수출 규모를 더욱 확대해 나갈 계획입니다.
특히, 세계 최대의 채소 재배국으로 부상한 중국은 2010년 중국 채소 생산면적 1,870만 헥타르, 채소 종자 소비량은 4만톤, 시장규모가 120억 위안에 달하며 2015년에는 중국 채소종자 시장 규모가 200억 위안을 넘을 것으로 확인되고 있습니다. 팬데믹시대 이후 각 정부들의 지속적인 부양책을 통해 경제 활성화가 식자재의 유통 및 소비현상을 부축이며 채소종자 소비가 지속될 것으로 보며 그에 따른 당사의 종자 수출액이 매년 6-13% 증가할 것으로 예상됩니다.
당사는 매년 국제종자교역회 6개국 6회, 해외 박람회 6개국 10회 이상 등에 참가하고 있으며, 향후 가능성 있는 남미지역의 종자교역회 SSA와 새로운 농업/원예 박람회 등에 참가해 종자 수출지역을 확대해 나아갈 계획입니다. 잠재적 가능성이 무궁무진한 아프리카, 중동, 북유럽, 남미 등에 영업 거점을 확보해 수출을 늘려나가고 있으며, 신규 시장 지역들의 연평균 수출 신장율 30-40% 이상을 목표로 하고 있습니다.
당사가 참가하고 있는 세계 각국 종자협회 교역회와 각종 박람회 현황은 아래와 같습니다.


[국제 종자 관련 협회 가입 활동 현황]


협회명
협회명 및 주관기관
개최시기
ASTA(American Seed Trade Association)
미국종자협회
매년 1월 참가
ISC(Indian Seed Congress)
인도종자총회
매년 2월 참가
AFSTA(African Seed Trade Association)
아프리카종자협회
매년 3월 참가
ISF(International Seed Federation)
국제종자협회
매년 5-6월 참가
SSA(Seed Association of the Americas)
남아메리카종자협회
매년 9월 참가
ESA(European Seed Association)
유럽종자협회
매년 10월 참가
APSA(The Asia & Pacific Seed Association)
아시아태평양종자협회
매년 11월 참가


당사는 농업 및 종자와 관련하여 세계적으로 영향력 있는 잡지 Vegetable Grower, Seed World, Gulf Agriculture, AS & PM 등에 지면 광고를 통하여 당사 제품을 홍보하고 있습니다. 당사는 당사 제품의 품질이 우수하며 여러 국가에 걸쳐 그 성능이 확인된 제품 위주 또는 신규 품종의 출시에 맞추어 전세계 농업인의 시선을 끌 수 있도록 적극적인 홍보 및 마케팅 전략을 펼쳐 나아가고 있습니다.

또한 당사는 중소기업청, KOTRA, 무역협회, 국립종자원 등에서 지원하는 무역, 종자 전문 인력 교육을 통하여 신규 및 기존 직원들의 무역, 종자 실무능력을 배양하고 있습니다. 또한 당사의 생명공학 육종연구소에 작물의 생리 및 재배 기술, 품종의 세부 특성 교육 등을 위탁하여 세계 어디에서든지 당사 품종의 우수성을 널리 알릴 수 있는 전문 인력을 육성하고 있습니다.


"""

In [29]:
client = OpenAI(api_key=gpt_api)

response = client.chat.completions.create(
    model="gpt-4.1-mini",   # 또는 gpt-4.1 / gpt-4o
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": content}
    ],
    temperature=0,
    max_tokens=8000
)

print(response.choices[0].message.content)

{
  "제22기": {
    "": {
      "양배추": {"금액": 4801, "비중": "18.53%"},
      "양파": {"금액": 2247, "비중": "8.67%"},
      "고추": {"금액": 1320, "비중": "5.09%"},
      "무": {"금액": 2162, "비중": "8.34%"},
      "옥수수": {"금액": 1115, "비중": "4.30%"},
      "청경채": {"금액": 494, "비중": "1.91%"},
      "파": {"금액": 861, "비중": "3.32%"},
      "기타작물": {"금액": 10755, "비중": "41.50%"},
      "기타상품": {"금액": 2159, "비중": "8.33%"},
      "합계": {"금액": 25914, "비중": "100.00%"}
    }
  }
}
